# CNN Architecture Experiments on MNIST

## 1. Title + Introduction

The MNIST dataset is a classic benchmark in computer vision that contains 70,000 grayscale images of handwritten digits from 0 through 9. Each image is 28 x 28 pixels, making the dataset small enough to train quickly while still being useful for testing image classification models.

Convolutional Neural Networks (CNNs) perform better than standard feedforward neural networks on image tasks because they preserve spatial structure and learn local visual features such as edges, curves, and strokes. A feedforward network flattens the image immediately, which removes important information about how nearby pixels relate to one another.

This notebook investigates how CNN performance changes when the number of filters and kernel sizes are adjusted. It also compares the best CNN architecture against a feedforward neural network and includes training visualizations, a performance summary table, test-set evaluation, and a confusion matrix.

## 2. Imports + Data Preparation

This section imports the required libraries, loads the MNIST dataset, normalizes pixel values to the range from 0 to 1, reshapes the images for convolutional layers, and prints the dataset shapes. A few sample digits are also displayed to give a quick visual check of the input data.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

plt.figure(figsize=(10, 3))
for index in range(8):
    plt.subplot(2, 4, index + 1)
    plt.imshow(x_train[index].reshape(28, 28), cmap="gray")
    plt.title(f"Label: {y_train[index]}")
    plt.axis("off")
plt.suptitle("Sample MNIST Digits", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Model Function

To keep the notebook organized and reusable, the CNN architecture is built with a function that accepts the number of filters and kernel sizes for the two convolutional layers. A second helper function handles model training and records elapsed training time, which makes the later experiments cleaner and easier to compare.

In [ ]:
def build_cnn(filters1, filters2, kernel1, kernel2):
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(filters1, kernel1, activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(filters2, kernel2, activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def build_feedforward():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def train_model(model, x_data, y_data, epochs, validation_split=0.2, batch_size=128):
    start_time = time.time()
    history = model.fit(
        x_data,
        y_data,
        epochs=epochs,
        validation_split=validation_split,
        batch_size=batch_size,
        verbose=1
    )
    elapsed_time = time.time() - start_time
    return history, elapsed_time


def evaluate_model(model, x_data, y_data):
    loss, accuracy = model.evaluate(x_data, y_data, verbose=0)
    return loss, accuracy


def architecture_label(filters1, filters2, kernel1, kernel2):
    return (
        f"Conv2D({filters1}, {kernel1}) -> MaxPool -> "
        f"Conv2D({filters2}, {kernel2}) -> MaxPool -> Dense(128)"
    )

## 4. Baseline CNN

The baseline CNN uses 32 filters in the first convolutional layer and 64 filters in the second, both with 3 x 3 kernels. This is a common starting architecture because the first layer learns simple local patterns while the second layer combines them into more complex digit features before classification.

In [ ]:
baseline_model = build_cnn(32, 64, (3, 3), (3, 3))
baseline_history, baseline_time = train_model(baseline_model, x_train, y_train, epochs=10)

baseline_train_accuracy = baseline_history.history["accuracy"][-1]
baseline_val_accuracy = baseline_history.history["val_accuracy"][-1]
baseline_test_loss, baseline_test_accuracy = evaluate_model(baseline_model, x_test, y_test)

print(f"Final training accuracy: {baseline_train_accuracy:.4f}")
print(f"Final validation accuracy: {baseline_val_accuracy:.4f}")
print(f"Test accuracy: {baseline_test_accuracy:.4f}")
print(f"Training time: {baseline_time:.2f} seconds")

## 5. Filter Experiments

This experiment changes the number of filters while keeping the kernel sizes fixed at 3 x 3. The goal is to measure whether giving the network more feature detectors improves validation accuracy enough to justify the added complexity and training cost.

Increasing filters usually helps the model learn richer visual features, especially in deeper layers where more detailed shape patterns can be captured. Diminishing returns tend to appear once the model is already learning the important digit structures and extra capacity only adds modest gains or mild overfitting.

In [ ]:
filter_configs = [(16, 32), (32, 64), (64, 128)]
filter_val_accuracies = {}
cnn_experiment_results = []

for filters1, filters2 in filter_configs:
    model = build_cnn(filters1, filters2, (3, 3), (3, 3))
    history, training_time = train_model(model, x_train, y_train, epochs=10)
    test_loss, test_accuracy = evaluate_model(model, x_test, y_test)

    final_train_accuracy = history.history["accuracy"][-1]
    final_val_accuracy = history.history["val_accuracy"][-1]

    filter_val_accuracies[(filters1, filters2)] = final_val_accuracy
    cnn_experiment_results.append({
        "Model": f"CNN Filters {filters1}-{filters2}",
        "Architecture": architecture_label(filters1, filters2, (3, 3), (3, 3)),
        "filters1": filters1,
        "filters2": filters2,
        "kernel1": (3, 3),
        "kernel2": (3, 3),
        "Train Accuracy": final_train_accuracy,
        "Validation Accuracy": final_val_accuracy,
        "Test Accuracy": test_accuracy,
        "Training Time": training_time
    })

print("Validation accuracy by filter configuration:")
print(filter_val_accuracies)

filter_results_df = pd.DataFrame(cnn_experiment_results)[[
    "Model", "Architecture", "Validation Accuracy", "Test Accuracy", "Training Time"
]]
filter_results_df["Validation Accuracy"] = filter_results_df["Validation Accuracy"].round(4)
filter_results_df["Test Accuracy"] = filter_results_df["Test Accuracy"].round(4)
filter_results_df["Training Time"] = filter_results_df["Training Time"].round(2)
filter_results_df

## 6. Kernel Experiments

This experiment keeps the number of filters fixed at 32 and 64 while changing the kernel sizes used by the convolutional layers. Kernel size affects how much of the image is examined at once, so it influences whether the model focuses more on fine local detail or broader patterns.

Smaller kernels such as 3 x 3 often perform well on MNIST because the digits are made of compact strokes and local edges. Larger kernels can sometimes help capture broader shapes, but they also introduce more parameters and may smooth out useful fine-grained information.

In [ ]:
kernel_configs = [((3, 3), (3, 3)), ((5, 5), (5, 5)), ((5, 5), (3, 3))]
kernel_val_accuracies = {}
kernel_experiment_results = []

for kernel1, kernel2 in kernel_configs:
    model = build_cnn(32, 64, kernel1, kernel2)
    history, training_time = train_model(model, x_train, y_train, epochs=10)
    test_loss, test_accuracy = evaluate_model(model, x_test, y_test)

    final_train_accuracy = history.history["accuracy"][-1]
    final_val_accuracy = history.history["val_accuracy"][-1]

    kernel_val_accuracies[(kernel1, kernel2)] = final_val_accuracy
    kernel_experiment_results.append({
        "Model": f"CNN Kernels {kernel1}-{kernel2}",
        "Architecture": architecture_label(32, 64, kernel1, kernel2),
        "filters1": 32,
        "filters2": 64,
        "kernel1": kernel1,
        "kernel2": kernel2,
        "Train Accuracy": final_train_accuracy,
        "Validation Accuracy": final_val_accuracy,
        "Test Accuracy": test_accuracy,
        "Training Time": training_time
    })

print("Validation accuracy by kernel configuration:")
print(kernel_val_accuracies)

kernel_results_df = pd.DataFrame(kernel_experiment_results)[[
    "Model", "Architecture", "Validation Accuracy", "Test Accuracy", "Training Time"
]]
kernel_results_df["Validation Accuracy"] = kernel_results_df["Validation Accuracy"].round(4)
kernel_results_df["Test Accuracy"] = kernel_results_df["Test Accuracy"].round(4)
kernel_results_df["Training Time"] = kernel_results_df["Training Time"].round(2)
kernel_results_df

## 7. Best Model + Training Visualization

The best CNN architecture is selected using validation accuracy from all CNN experiments. After selecting that architecture, the model is retrained for 15 epochs so the learning curves can be visualized and interpreted.

If the training accuracy continues rising while validation accuracy stalls or validation loss increases, the model may be overfitting. If both training and validation metrics improve together and remain close, the model is learning useful patterns and generalizing well.

In [ ]:
all_cnn_results = cnn_experiment_results + kernel_experiment_results
best_config = max(all_cnn_results, key=lambda item: item["Validation Accuracy"])

best_filters1 = best_config["filters1"]
best_filters2 = best_config["filters2"]
best_kernel1 = best_config["kernel1"]
best_kernel2 = best_config["kernel2"]

print("Best architecture based on validation accuracy:")
print(best_config["Architecture"])
print(f"Validation accuracy from experiment phase: {best_config['Validation Accuracy']:.4f}")

best_model = build_cnn(best_filters1, best_filters2, best_kernel1, best_kernel2)
best_history, best_time = train_model(best_model, x_train, y_train, epochs=15)
best_test_loss, best_test_accuracy = evaluate_model(best_model, x_test, y_test)

print(f"Best model test accuracy: {best_test_accuracy:.4f}")
print(f"Best model training time: {best_time:.2f} seconds")
best_model.summary()

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(best_history.history["accuracy"], label="Training Accuracy")
plt.plot(best_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Best CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(best_history.history["loss"], label="Training Loss")
plt.plot(best_history.history["val_loss"], label="Validation Loss")
plt.title("Best CNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Feedforward Network

A feedforward neural network is trained as a comparison baseline that does not use convolution. This highlights how much benefit comes from preserving spatial relationships and learning local image features directly.

In [ ]:
feedforward_model = build_feedforward()
feedforward_history, feedforward_time = train_model(feedforward_model, x_train, y_train, epochs=10)
feedforward_test_loss, feedforward_test_accuracy = evaluate_model(feedforward_model, x_test, y_test)

feedforward_train_accuracy = feedforward_history.history["accuracy"][-1]
feedforward_val_accuracy = feedforward_history.history["val_accuracy"][-1]

print(f"Final training accuracy: {feedforward_train_accuracy:.4f}")
print(f"Final validation accuracy: {feedforward_val_accuracy:.4f}")
print(f"Test accuracy: {feedforward_test_accuracy:.4f}")
print(f"Training time: {feedforward_time:.2f} seconds")

## 9. Performance Comparison Table

The final comparison table summarizes the baseline CNN, the best-performing CNN, and the feedforward neural network using test accuracy and training time. This makes it easier to compare model quality and computational cost side by side.

CNNs usually outperform feedforward networks on image tasks because they preserve spatial relationships and build features hierarchically from local pixel patterns. A feedforward network can still learn useful signals, but it loses much of the image structure when the input is flattened at the beginning.

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Baseline CNN",
        "Architecture": architecture_label(32, 64, (3, 3), (3, 3)),
        "Accuracy": baseline_test_accuracy,
        "Training Time": baseline_time
    },
    {
        "Model": "Best CNN",
        "Architecture": architecture_label(best_filters1, best_filters2, best_kernel1, best_kernel2),
        "Accuracy": best_test_accuracy,
        "Training Time": best_time
    },
    {
        "Model": "Feedforward NN",
        "Architecture": "Flatten -> Dense(128) -> Dense(64) -> Dense(10)",
        "Accuracy": feedforward_test_accuracy,
        "Training Time": feedforward_time
    }
])

comparison_df["Accuracy"] = comparison_df["Accuracy"].round(4)
comparison_df["Training Time"] = comparison_df["Training Time"].round(2)
comparison_df

## 10. Final Reflection

CNNs would improve a final project if the data includes images because they are built to recognize spatial patterns efficiently. RNNs or other sequence-oriented models would be more appropriate for time-series or ordered text data because they can model dependencies across steps in a sequence. If a final project is mainly tabular data, CNNs and RNNs may not provide much benefit because the features usually do not have strong spatial or sequential structure. In that case, dense networks or traditional machine learning models are often a better fit.

## 11. Optional Extensions

To make the notebook more complete and grading-ready, the final section includes a full test-set evaluation, the best model summary, and a confusion matrix. These additions help show not only how accurate the model is overall, but also where it still makes classification mistakes.

In [ ]:
best_predictions = best_model.predict(x_test, verbose=0)
best_predicted_labels = np.argmax(best_predictions, axis=1)

confusion_matrix = tf.math.confusion_matrix(y_test, best_predicted_labels, num_classes=10).numpy()

plt.figure(figsize=(8, 6))
plt.imshow(confusion_matrix, cmap="Blues")
plt.title("Confusion Matrix for Best CNN")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(np.arange(10))
plt.yticks(np.arange(10))

for row in range(confusion_matrix.shape[0]):
    for column in range(confusion_matrix.shape[1]):
        plt.text(column, row, confusion_matrix[row, column], ha="center", va="center", fontsize=8)

plt.colorbar()
plt.tight_layout()
plt.show()

print(f"Best model test loss: {best_test_loss:.4f}")
print(f"Best model test accuracy: {best_test_accuracy:.4f}")